## 这是一个简单的社交机器人架构示例

### 导入必要的库

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import wrap_model_call,dynamic_prompt
from langchain.messages import HumanMessage,AIMessage,ToolMessage,SystemMessage
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_deepseek import ChatDeepSeek
import os
from dotenv import load_dotenv


### 预先使用全局变量定义必要参数
为了安全，大模型API密钥存在环境配置里，而不是直接写在代码里

In [ ]:
load_dotenv()


SYSTEM_PROMPT ="你是一个社交机器人,现在你的人格设定是NBA球员科比·布莱恩特的粉丝"
DEEPSEEK_API_KEY =os.getenv("DEEPSEEK_API_KEY")


### 大模型接入
这里使用的是DeepSeek的模型

In [ ]:
llm_deepseek = ChatDeepSeek(
    model="deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
)

### Agent实例创建


In [ ]:
ag=create_agent(
    model=llm_deepseek,
    system_prompt=SYSTEM_PROMPT,
    tools=[],
    middleware=[],
    checkpointer=InMemorySaver(),
)

### Agent调用示例
在config参数里开启了线程级记忆，每次运行时无需在代码里维护对话历史，底层会自动实现上下文记忆
每次仅传入最新的用户输入

In [ ]:
msg = []
config={"configurable":{"thread_id":"1"}}

while True:
    user_input = input("User: ")
    if user_input == "exit":
        break
    msg.append(HumanMessage(content=user_input))
    response = ag.invoke({"messages":msg},config=config)
    msg=[]
    print("User:", user_input)
    print("Agent:", response["messages"][-1].content)